# 12 - Historical Companies House snapshots and the company-month panel

So far this project has run off a single Companies House snapshot (the June 2026 bulk file). That is enough to describe a company *today*, but it tells me nothing about how a company is *changing*, and change is the actual buying signal I care about (a jump in charges, a slide into distress, a step up in size). It also gives me nothing to predict, so there is no way to fit a model or run SHAP.

This notebook fixes that in two steps:

1. **Acquire** the historical bulk snapshots, so I have a monthly time series.
2. **Build** a company-month panel from them: one row per (company, month).

Companies House only links the current month on its download page, but the older monthly zips stay on the server and are reachable by direct URL. I confirmed by probing that 33 months exist, Oct 2023 to Jul 2026.

One thing to flag up front: **June 2025 is genuinely missing** from the server. That is not a bug in my code, it is a real hole, and I handle it in the next notebook when I compute deltas on a calendar-aware month spine (so a gap never silently corrupts a 3-month or 12-month difference).

I keep the logic in `src/data/ch_bulk.py` and `src/features/panel.py` and just drive it from here, the same way notebooks 10 and 11 lean on `src/features`.

## Setup

I point Python at the repo root so `from src.data import ch_bulk` resolves, exactly like the earlier notebooks do. The heavy libraries for the modelling steps (duckdb, lightgbm, shap, scikit-learn) are already installed in the project `.venv`; nothing new is needed just to download.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `src` is visible (src is a namespace package).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.data import ch_bulk

print("repo root:", REPO_ROOT)
print("snapshots dir:", (REPO_ROOT / ch_bulk.SNAPSHOT_DIR).resolve())
print("manifest months:", len(ch_bulk.MANIFEST))

# Step 1 - Acquire the snapshots

## The manifest

`ch_bulk.MANIFEST` is the verified list of 33 snapshot dates. The filenames are not reliably the 1st of the month (some months land on the 4th, 7th, 2nd), so I probed days 01 to 15 of each month once and hard-coded the dates that actually resolved. Day to day I do not want to re-hit the server just to know the dates, so the module keeps the verified list and only probes on demand.

The cell below shows the dates and makes the June 2025 gap visible.

In [ ]:
# Walk every month from Oct 2023 to Jul 2026 and mark which ones we have.
y, m = 2023, 10
while (y, m) <= (2026, 7):
    ym = f"{y}-{m:02d}"
    date = next((d for d in ch_bulk.MANIFEST if d.startswith(ym)), None)
    print(f"{ym}  {'-> ' + date if date else 'MISSING (real hole on CH server)'}")
    m += 1
    if m > 12:
        y, m = y + 1, 1

## (Optional) re-probe the server

If I ever want to refresh the manifest (for example a new month has published, or I am checking whether an older file has aged out), I can re-run the probe. It sends a cheap HEAD request per candidate day and takes the first hit per month. I leave this off by default so the notebook does not spam the CH server on every run.

In [ ]:
REPROBE = False
if REPROBE:
    months = [d[:7] for d in ch_bulk.MANIFEST]  # or any list of 'YYYY-MM'
    live = ch_bulk.probe_snapshot_dates(months)
    print(f"{len(live)} live dates found")
    for d in live:
        print(" ", d)
else:
    print("Re-probe skipped; using the verified manifest.")

## Download

Now the actual pull. Notes on how this behaves:

- **Where:** everything goes to `data/raw/snapshots/`. That sits under `data/`, which is fully gitignored, so these ~15 GB of zips never get pushed to the online repo.
- **Resumable:** any zip already fully on disk is skipped, so I can re-run this cell freely and it only fetches what is missing. Each file is streamed to a `.part` and only renamed on success, so an interrupted download never leaves a half-file masquerading as complete.
- **Parallel:** 4 downloads at a time, which is a reasonable balance between speed and being polite to the server.
- **Retries:** each file gets up to 3 attempts before it is reported as FAILED.

I keep the zips permanently. Once Companies House ages these months off the server they are the only copy I have, and re-deriving the panel with a different column set later then costs zero downloads.

In [ ]:
# This is the ~15 GB pull. Safe to re-run: it skips anything already downloaded.
results = ch_bulk.download_snapshots(dest_dir=REPO_ROOT / ch_bulk.SNAPSHOT_DIR)

ok = sum(1 for s in results.values() if s.startswith(("downloaded", "skipped")))
print(f"\n{ok}/{len(results)} snapshots present.")
failed = {d: s for d, s in results.items() if s.startswith("FAILED")}
if failed:
    print("Failures (re-run the cell to retry):")
    for d, s in failed.items():
        print(" ", d, s)

## Verify what I have

A quick tally of which manifest dates are on disk, so I can confirm the acquisition is complete before moving on to building the panel.

In [ ]:
status = ch_bulk.snapshot_status(dest_dir=REPO_ROOT / ch_bulk.SNAPSHOT_DIR)
present = [d for d, ok in status.items() if ok]
missing = [d for d, ok in status.items() if not ok]
print(f"present: {len(present)}/{len(status)}")
if missing:
    print("still missing:", missing)

total_gb = sum(
    (REPO_ROOT / ch_bulk.SNAPSHOT_DIR / f"BasicCompanyDataAsOneFile-{d}.zip").stat().st_size
    for d in present
) / (1 << 30)
print(f"total on disk: {total_gb:,.1f} GB")

---

# Step 2 - Build the company-month panel

Now I turn 33 loose monthly CSVs into one tidy company-month panel: one row per (company, month). That is the shape I need for deltas and for self-labelled targets.

Two design decisions here are worth spelling out, because they are the ones that would quietly ruin the modelling if I got them wrong.

**1. Two passes, not one.** My first instinct was to just replay notebook 1's filter on every month. That is wrong. A company can be re-coded out of my target SIC sectors without dying, and if I filtered each month independently that company would simply vanish from the panel, looking exactly like a dissolution. So instead:

- **Pass 1 (universe):** collect every `CompanyNumber` that matches my sectors in *any* month.
- **Pass 2 (extract):** emit a row for every universe member present in a month, whatever its status or current sector.

Absence from the panel now means one thing only: gone from the register.

**2. Keep all statuses.** Notebook 1 filters to `CompanyStatus == "Active"`, which made sense there. If I replayed that per month I would delete every failure event, and I would end up with a panel in which no company ever fails, so a credit-risk model would have nothing to learn from. The insolvent firms are not prospects, but they are exactly the *labels* I need. So I keep them in the panel and filter `is_active` at scoring time instead.

There is also a staging step in the middle, purely for speed: each 2.7 GB CSV gets extracted and parsed exactly once into a stage parquet, then both passes read the cheap parquet. The extracted CSV is deleted as soon as its stage partition is written (the zip is the archive, the CSV is scratch).

## The sector mapping, lifted verbatim from notebook 1

`panel.py` carries notebook 1's `SEGMENT_MAP`, `FAST_GROWTH_CODES` and `TARGET_SECTIONS` unchanged. I copied them rather than reimplementing, because the whole point of the parity check further down is to prove the historical replay matches notebook 1 exactly. Two subtleties I deliberately preserved:

- Fast-growth codes are assigned **first**, so they take priority over whatever SIC section they sit in.
- Within a row, the **first non-null SIC match wins in column order** (`SicText_1` through `SicText_4`). That is notebook 1's `combine_first` behaviour, so I kept it even though a row-level fast-growth priority might arguably be nicer. Parity beats taste here.

In [ ]:
from collections import Counter

from src.features import panel

sector_map = panel.load_sector_map(REPO_ROOT / panel.SIC_PATH)
print(f"SIC codes mapped: {len(sector_map):,}")
for label, n in Counter(sector_map.values()).most_common():
    print(f"  {label:<35} {n:>4}")

## Run the build

`build_panel` does all three stages: stage every snapshot, union the universe, extract the panel. It is resumable at every step (any stage or panel partition already on disk is skipped), so I can interrupt and re-run this cell freely.

One point-in-time detail that matters a lot: pass 2 calls `ch_static.add_static_features(df, today=<the real snapshot date>)`. Passing `today` explicitly is what makes `company_age_years`, `accounts_overdue` and `accounts_stale` reflect what was true *then*, rather than what is true whenever I happen to run the code. Without it, a feature computed against "now" would be quietly leaking the future into every historical row.

Note the partitions are keyed on the **month start** even where the file date is not the 1st (Oct 2023 is the 4th, Feb 2024 the 7th). The true file date is kept alongside as `source_date` and is what the point-in-time features use. Keying on the month is what lets me join a clean calendar spine in the next notebook.

This takes roughly 10 to 15 minutes on my machine.

In [ ]:
counts = panel.build_panel(threads=22)

print(f"\npartitions: {len(counts)}")
print(f"total rows: {sum(counts.values()):,}")

## Verification 1 - parity against notebook 1 (the important one)

This is the check that decides whether any of the rest is trustworthy. If I take the panel's June 2026 partition and filter it back down to `Active` and in-sector, I should land exactly on `filtered_bb_sme_sectors.csv`: **1,372,321 rows**, the same `CompanyNumber` set, and the same sector counts (Tech/legal 837,066, Fast growth 320,670, Manufacturing 214,585).

If this matches, my historical replay is faithful to notebook 1, and the other 32 months are built by exactly the same code path.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")

part = REPO_ROOT / panel.PANEL_DIR / "snapshot_date=2026-06-01" / "part.parquet"

# The panel keeps all statuses, so I filter back to NB01's view to compare like for like.
replay = con.execute(f"""
    SELECT sector, count(*) AS n
    FROM read_parquet('{part.as_posix()}')
    WHERE is_active AND sector IS NOT NULL
    GROUP BY sector ORDER BY n DESC
""").df()
print(replay.to_string(index=False))
print(f"\nreplay total: {replay['n'].sum():,}")

EXPECTED = {
    "Technology, legal & professional": 837_066,
    "Fast growth & emerging": 320_670,
    "Manufacturing": 214_585,
}
got = dict(zip(replay["sector"], replay["n"]))
assert got == EXPECTED, f"sector counts differ: {got} vs {EXPECTED}"
assert replay["n"].sum() == 1_372_321
print("\nSector counts and total match notebook 1.")

In [ ]:
# Counts matching is necessary but not sufficient, so I check the actual CompanyNumber sets.
nb01 = pd.read_csv(
    REPO_ROOT / "data/processed/filtered_bb_sme_sectors.csv",
    usecols=lambda c: c.strip() == "CompanyNumber",
    dtype=str,
)
nb01.columns = nb01.columns.str.strip()
nb01_set = set(nb01["CompanyNumber"].str.strip())

replay_set = set(con.execute(f"""
    SELECT "CompanyNumber" FROM read_parquet('{part.as_posix()}')
    WHERE is_active AND sector IS NOT NULL
""").df()["CompanyNumber"])

print(f"notebook 1 : {len(nb01_set):,}")
print(f"panel replay: {len(replay_set):,}")
print(f"missing from replay: {len(nb01_set - replay_set):,}")
print(f"extra in replay    : {len(replay_set - nb01_set):,}")
assert nb01_set == replay_set, "CompanyNumber sets differ"
print("\nCompanyNumber sets are identical. The replay is faithful.")

## Verification 2 - point-in-time correctness

If `today=snapshot_date` was honoured, then walking a single company forward through the snapshots should show `company_age_years` **increasing**, roughly one year per twelve months. If I had left `ch_static` to default to `Timestamp.today()`, the age would instead be flat and identical across every month, which is the tell-tale sign of a leaked "now".

In [ ]:
glob_all = (REPO_ROOT / panel.PANEL_DIR / "**" / "*.parquet").as_posix()

# Pick one long-lived company and walk it through time.
probe = con.execute(f"""
    SELECT "CompanyNumber", snapshot_date, company_age_years, "CompanyStatus"
    FROM read_parquet('{glob_all}')
    WHERE "CompanyNumber" = (
        SELECT "CompanyNumber" FROM read_parquet('{part.as_posix()}')
        WHERE is_active AND company_age_years > 20 LIMIT 1
    )
    ORDER BY snapshot_date
""").df()
print(probe.to_string(index=False))

ages = probe["company_age_years"].tolist()
assert ages == sorted(ages), "company_age_years should increase with snapshot_date"
assert len(set(ages)) > 1, "age is constant across months, so Timestamp.today() leaked in"
print("\nAge moves with the snapshot date, so the features are point-in-time correct.")

## What the panel looks like

A quick shape check: rows per month, and how the status mix evolves. The row count per month should drift up gently (new incorporations joining the universe) and the non-Active share should grow over time as companies fail, which is precisely the signal the credit-risk model will key off. The June 2025 gap should be visible as a missing month, not as a silently interpolated one.

In [ ]:
shape = con.execute(f"""
    SELECT snapshot_date,
           count(*) AS n_rows,
           sum(CASE WHEN is_active THEN 1 ELSE 0 END) AS active,
           round(100.0 * sum(CASE WHEN NOT is_active THEN 1 ELSE 0 END) / count(*), 2) AS pct_not_active,
           sum(CASE WHEN sector IS NULL THEN 1 ELSE 0 END) AS recoded_out_of_sector
    FROM read_parquet('{glob_all}')
    GROUP BY snapshot_date ORDER BY snapshot_date
""").df()
print(shape.to_string(index=False))

The `recoded_out_of_sector` column is the two-pass design earning its keep: those are companies still alive on the register whose SIC no longer maps to one of my sectors. A naive per-month filter would have dropped them and I would have mistaken a re-coding for a death.

Likewise, the rising `pct_not_active` is exactly what I was protecting when I decided to keep every status. Under notebook 1's Active-only filter that column would read 0.00 in every single month, and there would be no failure for a credit-risk model to learn from.

## Next

The panel is built and verified against notebook 1. Notebook 13 picks it up from here: join a proper calendar month spine (so the June 2025 hole leaves deltas NULL rather than silently spanning four months), compute the ~25 delta features, join the Contracts Finder history as-of each month, and build the three self-labelled targets.